In [ ]:
# 04 — Horizon Test
Goal: Test if the model can predict 60 minutes into the future, not just 15 minutes.

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

# Load the data again
aau = pd.read_csv("../data/raw/AAU_temperature_and_power_use.csv")
aau['Time [Date/Time]'] = pd.to_datetime(aau['Time [Date/Time]'])

cols_to_keep = ['Time [Date/Time]', 'Temperature mod 1 ch 1', 'Power Ch 1 (W)', 'Power Ch 2 (W)', 'Power Ch 3 (W)']
df = aau[cols_to_keep].copy()
df = df.set_index('Time [Date/Time]').resample('1min').mean().reset_index()

df = df.rename(columns={
    'Time [Date/Time]': 'time', 'Temperature mod 1 ch 1': 'temp',
    'Power Ch 1 (W)': 'power_1', 'Power Ch 2 (W)': 'power_2', 'Power Ch 3 (W)': 'power_3'
})

df['hour'] = df['time'].dt.hour
df['minute'] = df['time'].dt.minute
df['regime'] = (df['time'] >= '2024-09-28').astype(int)

# --- THE CHANGE: Target is now 60 minutes ahead ---
df['temp_target_60m'] = df['temp'].shift(-60)

# Lags (Keep them the same)
df['temp_lag_1m']  = df['temp'].shift(1)
df['temp_lag_5m']  = df['temp'].shift(5)
df['temp_lag_15m'] = df['temp'].shift(15)
df['temp_lag_60m'] = df['temp'].shift(60)
df['power_1_lag_1m'] = df['power_1'].shift(1)
df['power_1_lag_5m'] = df['power_1'].shift(5)
df['temp_rolling_mean_15m'] = df['temp'].rolling(window=15).mean()

df_clean = df.dropna().reset_index(drop=True)

In [3]:
split_date = pd.to_datetime('2024-09-28')
train = df_clean[df_clean['time'] < split_date]
test  = df_clean[df_clean['time'] >= split_date]

features = ['power_1', 'power_2', 'power_3', 'hour', 'minute', 'regime', 
            'temp_lag_1m', 'temp_lag_5m', 'temp_lag_15m', 'temp_lag_60m', 
            'power_1_lag_1m', 'power_1_lag_5m', 'temp_rolling_mean_15m']

X_train, y_train = train[features], train['temp_target_60m']
X_test, y_test = test[features], test['temp_target_60m']

model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=7, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

# Evaluate
mae_60 = mean_absolute_error(y_test, predictions)
baseline_mae_60 = mean_absolute_error(y_test, test['temp'])

print(f"60-Min MAE: {mae_60:.3f} °C")
print(f"60-Min Baseline MAE: {baseline_mae_60:.3f} °C")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000841 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2634
[LightGBM] [Info] Number of data points in the train set: 16202, number of used features: 12
[LightGBM] [Info] Start training from score 23.528528
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain